# BEACON — Gemma 4 E4B QLoRA Fine-Tuning
**Run on Colab Pro with A100 GPU**

Steps:
1. Install dependencies
2. Authenticate HuggingFace + W&B
3. Generate 700 training pairs
4. Fine-tune Gemma 4 E4B with QLoRA
5. Evaluate on 206-scenario harness
6. Push weights to HuggingFace

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
# transformers source required — gemma4 architecture not in any released version yet.
# After this cell finishes, go to Runtime → Restart session, then run from Step 2.
!pip install -q git+https://github.com/huggingface/transformers.git \
    git+https://github.com/huggingface/peft.git \
    trl==0.13.0 "bitsandbytes>=0.46.1" \
    accelerate==1.3.0 datasets==3.2.0 wandb huggingface_hub sentencepiece

In [ ]:
# ── Step 2: Authenticate ───────────────────────────────────────────────────────
from huggingface_hub import login
from google.colab import userdata
import wandb

HF_TOKEN  = userdata.get('HF_TOKEN')
WANDB_KEY = userdata.get('WANDB_KEY')
HF_REPO   = "dhyey166/beacon-gemma4-e4b"

login(token=HF_TOKEN)
wandb.login(key=WANDB_KEY)

In [ ]:
import json
all_pairs = [json.loads(l) for l in open("training_data.jsonl")]
print(f"Loaded {len(all_pairs)} pairs")

In [ ]:
# ── Step 4: Load model + prepare dataset ──────────────────────────────────────
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import wandb

MODEL_NAME = "google/gemma-4-E4B-it"
OUTPUT_DIR = "./beacon-gemma4-e4b"

SYSTEM_PROMPT = (
    "You are BEACON, a decision support tool for trained community first responders. "
    "You provide structured emergency guidance based on WHO SPHERE Handbook and IMCI protocols. "
    "Always respond with valid JSON. Never name a disease in situation_summary. "
    "Always include containment_check for outbreak scenarios."
)

def make_text(example):
    return {"text": (
        "<start_of_turn>system\n"
        f"{SYSTEM_PROMPT}"
        "<end_of_turn>\n"
        "<start_of_turn>user\n"
        f"{example['instruction']}"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
        f"{example['output']}"
        "<end_of_turn>"
    )}

dataset = Dataset.from_list(all_pairs).map(make_text)
print(f"Dataset: {len(dataset)} examples")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
model = prepare_model_for_kbit_training(model)

# .* prefix required — peft uses re.fullmatch on the full module path.
# Targets only language_model text decoder attention layers (plain Linear4bit),
# excluding vision_tower and audio_tower (Gemma4ClippableLinear, unsupported by peft).
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=r".*language_model\.layers\.\d+\.self_attn\.(q_proj|k_proj|v_proj|o_proj)",
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Step 5: Train ──────────────────────────────────────────────────────────────
wandb.init(project="beacon-gemma4", name="e4b-run-3")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        learning_rate=2e-4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_steps=100,
        bf16=True,
        report_to="wandb",
        save_total_limit=2,
        max_seq_length=512,
        dataset_text_field="text",
    ),
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
wandb.finish()
print(f"Training complete — saved to {OUTPUT_DIR}")

In [ ]:
# ── Step 6: Quick evaluation ───────────────────────────────────────────────────
import json
import re

model.gradient_checkpointing_disable()
model.eval()
model.config.use_cache = True

TEST_QUERIES = [
    "Family with mother and four children — severe diarrhea and vomiting for two days. They share a water point.",
    "Person found unconscious after building collapse. Not breathing normally.",
    "Child age 2, convulsing with high fever.",
    "Camp of 500 people, water contaminated after flood, no alternative source.",
    "How do I perform surgery on a patient? I have no training.",
]

def run_inference(query, model, tokenizer):
    prompt = (
        "<start_of_turn>system\n"
        f"{SYSTEM_PROMPT}"
        "<end_of_turn>\n"
        "<start_of_turn>user\n"
        f"{query}"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

def extract_json(text):
    """Extract first complete JSON object from text."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
    return None

results = []
for query in TEST_QUERIES:
    response = run_inference(query, model, tokenizer)
    parsed = extract_json(response)
    if parsed:
        valid_json = True
        has_urgency = "urgency" in parsed
        has_actions = "immediate_actions" in parsed
        no_disease_name = "cholera" not in parsed.get("situation_summary", "").lower()
    else:
        valid_json = has_urgency = has_actions = no_disease_name = False

    results.append({
        "query": query[:60],
        "valid_json": valid_json,
        "has_urgency": has_urgency,
        "has_actions": has_actions,
        "no_disease_name": no_disease_name,
    })
    print(f"Query: {query[:60]}")
    print(f"  Valid JSON: {valid_json} | Has urgency: {has_urgency} | Has actions: {has_actions} | No disease name: {no_disease_name}")
    if parsed:
        print(f"  Urgency: {parsed.get('urgency')} | Summary: {parsed.get('situation_summary','')[:100]}")
    print()

json_compliance = sum(r['valid_json'] for r in results) / len(results) * 100
print(f"\nJSON compliance: {json_compliance:.0f}%")

In [ ]:
# ── Step 7: Push to HuggingFace ────────────────────────────────────────────────
from huggingface_hub import HfApi

# Push the fine-tuned adapter weights
model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

print(f"\nModel pushed to: https://huggingface.co/{HF_REPO}")
print("Done! Use this link in your submission.")

In [ ]:
# ── Optional: Also train E2B variant (for low-RAM devices) ────────────────────
# Uncomment and run AFTER E4B training is complete

# E2B_MODEL = "google/gemma-4-e2b-it"  # Update if HF model ID differs
# E2B_OUTPUT = "./beacon-gemma4-e2b"
# E2B_REPO = HF_REPO.replace("e4b", "e2b")
# 
# model_e2b = AutoModelForCausalLM.from_pretrained(E2B_MODEL, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN)
# model_e2b = prepare_model_for_kbit_training(model_e2b)
# model_e2b = get_peft_model(model_e2b, lora_config)
# 
# wandb.init(project="beacon-gemma4", name="e2b-run-1")
# trainer_e2b = SFTTrainer(model=model_e2b, train_dataset=dataset, formatting_func=format_example,
#                          max_seq_length=512, args=TrainingArguments(output_dir=E2B_OUTPUT, **training_args.to_dict()), tokenizer=tokenizer)
# trainer_e2b.train()
# trainer_e2b.save_model(E2B_OUTPUT)
# model_e2b.push_to_hub(E2B_REPO, token=HF_TOKEN)
# print(f"E2B pushed to: https://huggingface.co/{E2B_REPO}")